In [1]:
import sys
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

sys.path.append("..")
from data.base import get_connection

print("Dependencies loaded successfully for Notebook 07b.")

# Define structural regime dates
REGIME_BOUNDS = {
    'pre_covid_end': '2020-02-29',
    'lockdown_end': '2021-06-30',
    'pre_24_7_end': '2022-07-31',
    'end_study': '2026-06-30'
}

def assign_regime(date_series):
    dt = pd.to_datetime(date_series)
    conditions = [
        dt <= REGIME_BOUNDS['pre_covid_end'],
        (dt > REGIME_BOUNDS['pre_covid_end']) & (dt <= REGIME_BOUNDS['lockdown_end']),
        (dt > REGIME_BOUNDS['lockdown_end']) & (dt <= REGIME_BOUNDS['pre_24_7_end']),
        dt > REGIME_BOUNDS['pre_24_7_end']
    ]
    choices = [
        '1_Pre_COVID', 
        '2_Lockdown', 
        '3_Rebound_Restricted', 
        '4_Era_24_7'
    ]
    return pd.Series(np.select(conditions, choices, default='1_Pre_COVID'), index=dt.index)

print("Regime assignment logic verified.")

Dependencies loaded successfully for Notebook 07b.
Regime assignment logic verified.


In [2]:
con = get_connection()

# Aggregate to weekly level to smooth weekend volatility and reduce panel size
query = """
    WITH cleaned_tickets AS (
        SELECT 
            CAST(t.issue_date AS DATE) AS date,
            CAST(t.violation_code AS INTEGER) AS violation_code,
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(
                            TRIM(UPPER(COALESCE(t.street_name, '') || ' ' || COALESCE(t.intersecting_street, ''))),
                            '\\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\\b', '', 'g'
                        ),
                        '\\(.*?\\)', '', 'g'
                    ),
                    '@', ' AND ', 'g'
                ),
                '\\s+', ' ', 'g'
            ) || ', NEW YORK, NY' AS search_query
        FROM tickets_all t
        WHERE CAST(t.violation_code AS INTEGER) IN (7, 36)
          AND t.issue_date >= '2016-01-01'
          AND t.street_name IS NOT NULL 
          AND t.intersecting_street IS NOT NULL
    ),
    weekly_tickets AS (
        SELECT 
            DATE_TRUNC('week', t.date) AS week_date,
            t.search_query,
            p.camera_type,
            COUNT(*) AS tickets_issued
        FROM cleaned_tickets t
        INNER JOIN read_csv_auto('../data/current_active_permanent_cameras_2026.csv') p 
            ON t.search_query = p.search_query
        GROUP BY 1, 2, 3
    ),
    weekly_crashes AS (
        SELECT 
            DATE_TRUNC('week', CAST(crash_date AS DATE)) AS week_date,
            search_query,
            COUNT(*) AS crash_count,
            SUM(COALESCE(number_of_persons_injured, 0)) AS injured_count,
            SUM(COALESCE(number_of_pedestrians_injured, 0)) AS ped_injured_count
        FROM camera_crash_buffers
        GROUP BY 1, 2
    ),
    weekly_backbone AS (
        SELECT 
            COALESCE(t.week_date, c.week_date) AS week_date,
            COALESCE(t.search_query, c.search_query) AS search_query,
            COALESCE(t.camera_type, 'Speed') AS camera_type,
            COALESCE(t.tickets_issued, 0) AS tickets_issued,
            COALESCE(c.crash_count, 0) AS crash_count,
            COALESCE(c.injured_count, 0) AS injured_count,
            COALESCE(c.ped_injured_count, 0) AS ped_injured_count
        FROM weekly_tickets t
        FULL OUTER JOIN weekly_crashes c 
          ON t.week_date = c.week_date AND t.search_query = c.search_query
    )
    SELECT 
        b.*,
        a.inferred_activation_date,
        -- Camera operational status in that week
        CASE 
            WHEN a.inferred_activation_date IS NOT NULL AND b.week_date >= a.inferred_activation_date 
            THEN 1 
            ELSE 0 
        END AS is_camera_active
    FROM weekly_backbone b
    LEFT JOIN camera_activation_dates a 
      ON b.search_query = a.search_query
    ORDER BY week_date, search_query
"""

try:
    print("Extracting weekly panel from DuckDB...")
    weekly_panel = con.sql(query).df()
    print(f"Extracted {len(weekly_panel):,} weekly camera-intersection records.")
finally:
    con.close()

weekly_panel['week_date'] = pd.to_datetime(weekly_panel['week_date'])
weekly_panel.head()

Extracting weekly panel from DuckDB...
Extracted 228,563 weekly camera-intersection records.


,week_date,search_query,camera_type,tickets_issued,crash_count,injured_count,ped_injured_count,inferred_activation_date,is_camera_active
0,2015-12-28,"10TH AVE AND LITTLE W 12TH ST, NEW YORK, NY",Speed,0,1,0.0,0.0,2019-11-25,0
1,2015-12-28,"112TH ST AND JAMAIC A AVE, NEW YORK, NY",Speed,0,1,1.0,0.0,2025-02-03,0
2,2015-12-28,"12TH AVE AND 34T H ST, NEW YORK, NY",Speed,0,1,0.0,0.0,2016-05-30,0
3,2015-12-28,"130TH ST AND N COND UIT AVE, NEW YORK, NY",Speed,0,1,0.0,0.0,2026-04-27,0
4,2015-12-28,"1ST AVE AND E 116TH ST, NEW YORK, NY",Speed,0,1,0.0,0.0,2025-08-04,0


In [3]:
# 1. Assign macro policy regimes
weekly_panel['regime'] = assign_regime(weekly_panel['week_date'])

# 2. Add log-transformed ticket volumes
weekly_panel['log_tickets'] = np.log1p(weekly_panel['tickets_issued'])

# 3. Add seasonality and calendar controls
weekly_panel['month'] = weekly_panel['week_date'].dt.month.astype(str)
weekly_panel['year'] = weekly_panel['week_date'].dt.year.astype(str)

print("Weekly Panel Overview by Regime:")
regime_stats = weekly_panel.groupby('regime').agg(
    total_weeks=('week_date', 'nunique'),
    obs_count=('week_date', 'count'),
    mean_tickets_per_week=('tickets_issued', 'mean'),
    mean_crashes_per_week=('crash_count', 'mean'),
    mean_injuries_per_week=('injured_count', 'mean')
).round(3)

print(regime_stats.to_string())

Weekly Panel Overview by Regime:
                      total_weeks  obs_count  mean_tickets_per_week  mean_crashes_per_week  mean_injuries_per_week
regime                                                                                                            
1_Pre_COVID                   218     112826                 14.031                  1.087                   0.344
2_Lockdown                     70      24503                 22.373                  0.721                   0.393
3_Rebound_Restricted           56      20497                 30.617                  0.755                   0.445
4_Era_24_7                    213      70737                 36.747                  0.671                   0.459


In [4]:
print("=== Model 1: Ticket Issuance Across Policy Regimes (Clustered OLS) ===")

# Reference category: 1_Pre_COVID
ticket_regime_model = smf.ols(
    """
    log_tickets ~ C(regime, Treatment(reference='1_Pre_COVID')) 
                  + C(camera_type) + C(month)
    """,
    data=weekly_panel
).fit(cov_type='cluster', cov_kwds={'groups': weekly_panel['search_query']})

print(ticket_regime_model.summary().tables[1])

# Calculate empirical percentage changes vs Pre-COVID baseline
for regime in ['2_Lockdown', '3_Rebound_Restricted', '4_Era_24_7']:
    coef = ticket_regime_model.params[f"C(regime, Treatment(reference='1_Pre_COVID'))[T.{regime}]"]
    pct = (np.exp(coef) - 1) * 100
    print(f"Ticket Volume Shift during {regime}: {pct:+.2f}%")

=== Model 1: Ticket Issuance Across Policy Regimes (Clustered OLS) ===
                                                                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                 3.5956      0.070     51.709      0.000       3.459       3.732
C(regime, Treatment(reference='1_Pre_COVID'))[T.2_Lockdown]               0.0093      0.025      0.367      0.714      -0.040       0.059
C(regime, Treatment(reference='1_Pre_COVID'))[T.3_Rebound_Restricted]     0.1467      0.025      5.761      0.000       0.097       0.197
C(regime, Treatment(reference='1_Pre_COVID'))[T.4_Era_24_7]               0.1957      0.031      6.291      0.000       0.135       0.257
C(camera_type)[T.Speed]                                                  -3.6524      0.087    -41.75

In [5]:
print("=== Model 2: Crash Risk & Camera Treatment Across Regimes (Poisson DiD) ===")

# Tests if camera deterrence was diluted during lockdown or strengthened under 24/7 law
crash_did_model = smf.glm(
    """
    crash_count ~ is_camera_active * C(regime, Treatment(reference='1_Pre_COVID')) 
                  + C(camera_type) + C(month)
    """,
    data=weekly_panel,
    family=sm.families.Poisson()
).fit(cov_type='cluster', cov_kwds={'groups': weekly_panel['search_query']})

print(crash_did_model.summary().tables[1])

# Interpret base camera effect and incremental regime shifts
base_cam_coef = crash_did_model.params['is_camera_active']
base_cam_pct = (np.exp(base_cam_coef) - 1) * 100
print(f"\nBaseline Camera Deterrence (Pre-COVID): {base_cam_pct:+.2f}%")

era_24_7_interact = crash_did_model.params["is_camera_active:C(regime, Treatment(reference='1_Pre_COVID'))[T.4_Era_24_7]"]
net_24_7_coef = base_cam_coef + era_24_7_interact
net_24_7_pct = (np.exp(net_24_7_coef) - 1) * 100
print(f"Net Camera Deterrence in 24/7 Enforcement Era: {net_24_7_pct:+.2f}%")

=== Model 2: Crash Risk & Camera Treatment Across Regimes (Poisson DiD) ===
                                                                                             coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                                 -1.0451      0.076    -13.838      0.000      -1.193      -0.897
C(regime, Treatment(reference='1_Pre_COVID'))[T.2_Lockdown]                               -0.1331      0.011    -12.534      0.000      -0.154      -0.112
C(regime, Treatment(reference='1_Pre_COVID'))[T.3_Rebound_Restricted]                     -0.1137      0.010    -10.899      0.000      -0.134      -0.093
C(regime, Treatment(reference='1_Pre_COVID'))[T.4_Era_24_7]                               -0.1281      0.010    -13.467      0.000      -0.147      -

In [6]:
print("=== Aggregate Citywide Weekly Time Series for Visualization & Break Test ===")

# Aggregate citywide weekly totals
city_weekly = weekly_panel.groupby(['week_date', 'regime']).agg(
    total_tickets=('tickets_issued', 'sum'),
    total_crashes=('crash_count', 'sum'),
    total_injuries=('injured_count', 'sum')
).reset_index().sort_values('week_date')

# Export aggregated series for plotting / reporting figures
os.makedirs('../data/output', exist_ok=True)
city_weekly.to_csv('../data/output/covid_weekly_regime_series.csv', index=False)
print("Saved weekly regime series to ../data/output/covid_weekly_regime_series.csv")

# Print structural comparison around March 2020 break
pre_covid_mean = city_weekly[city_weekly['regime'] == '1_Pre_COVID']['total_crashes'].mean()
lockdown_mean = city_weekly[city_weekly['regime'] == '2_Lockdown']['total_crashes'].mean()
pct_drop_lockdown = ((lockdown_mean - pre_covid_mean) / pre_covid_mean) * 100

print(f"\nPre-COVID Mean Weekly Crashes (Buffered): {pre_covid_mean:,.1f}")
print(f"Lockdown Mean Weekly Crashes (Buffered):  {lockdown_mean:,.1f}")
print(f"Aggregate Lockdown Collision Shift:      {pct_drop_lockdown:+.2f}%")

=== Aggregate Citywide Weekly Time Series for Visualization & Break Test ===
Saved weekly regime series to ../data/output/covid_weekly_regime_series.csv

Pre-COVID Mean Weekly Crashes (Buffered): 562.8
Lockdown Mean Weekly Crashes (Buffered):  252.5
Aggregate Lockdown Collision Shift:      -55.13%
